# DSC 180A checkpoint: GPU fallback for the remaining CIFAR experiments

Select **Runtime → Change runtime type → T4 GPU** (a free GPU, if available), then **Run all**. This runs the required 1,000-image guidance sweep and both 20,000-update subset-training experiments. MNIST results are already in the repository. The final cell downloads the figures, metrics, logs, and report source for review. This does not require a paid runtime.


In [ ]:
import os, pathlib, subprocess, sys, torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
repo = pathlib.Path('/content/DSC-180A-Checkpoint')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/aathijmuthu/DSC-180A-Checkpoint.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
os.environ['PYTHON'] = sys.executable
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONUNBUFFERED'] = '1'


## Run the required CIFAR work

The script prints progress, saves training state every 500 updates, and continues other deliverables if one task fails. Re-running this cell resumes saved training and reuses completed sweep settings. Keep this runtime connected until the export finishes.


In [ ]:
subprocess.run(['bash', 'scripts/run_dsmlp.sh', 'cifar'], check=True)


## Download results

Run this cell even if an earlier experiment failed: it exports all available results. Checkpoints are excluded to keep the download small. Return this ZIP to the local project so the report can be updated and compiled.


In [ ]:
import zipfile
from google.colab import files
subprocess.run([sys.executable, '-m', 'checkpoint.summarize'], check=True)
archive = pathlib.Path('/content/dsc180-cifar-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['artifacts', 'logs', 'report']:
        for path in pathlib.Path(folder).rglob('*'):
            if path.is_file() and path.suffix in {'.json', '.csv', '.png', '.txt', '.log', '.tex', '.npz'}:
                z.write(path, str(path))
    for name in ['RESULTS.md', 'notes.md', 'README.md']:
        z.write(name, name)
files.download(str(archive))
